In [2]:
!pip install sktime

import os
import numpy as np
import pandas as pd
import scipy.io
import librosa
import kagglehub
import time
from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeClassifierCV
from IPython.display import Audio, display
from sktime.classification.kernel_based import RocketClassifier
from sktime.transformations.panel.rocket import Rocket
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

zsh:1: command not found: pip


/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# ==========================================
# 1. CARGA Y FILTRADO DEL DATASET (ESC-50)
# ==========================================
print("Descarga del dataset desde Kaggle...")
dataset_root_path = kagglehub.dataset_download("mmoreaux/environmental-sound-classification-50")

# Imprimo los nombres de los archivos descargados
print(f"Contenido de la carpeta descargada por Kagglehub: {os.listdir(dataset_root_path)}")

# Guardo la carpeta de metadata
metadata_path = os.path.join(dataset_root_path, 'esc50.csv')

if os.path.exists(metadata_path):
    df = pd.read_csv(metadata_path)
    print("Archivo de metadatos cargado correctamente.")
else:
    raise FileNotFoundError(f"No se encontró el archivo esc50.csv en la ruta: {metadata_path}")

# Mis clases
mis_clases = ['alarm', 'door_bell', 'cat', 'crying_baby', 'dog', 'shouting']
df_filtrado = df[df['category'].isin(mis_clases)].copy()

# Muestras con las etiquetas que busco
print(f"Total de muestras encontradas para tus 6 clases en el CSV: {len(df_filtrado)}")


Descarga del dataset desde Kaggle...
Contenido de la carpeta descargada por Kagglehub: ['esc50.csv', 'audio', 'utils.py', 'utils2.py', 'bc_utils.py']
Archivo de metadatos cargado correctamente.
Total de muestras encontradas para tus 6 clases en el CSV: 120


In [8]:
# ==================================================
# 4. CARGA Y CONCATENACIÓN DE BABBLE NOISE - DINÁMICO
# ==================================================
import os
import scipy.io
import numpy as np
import librosa
import h5py
import kagglehub

print("🔍 Solicitando la ruta exacta al gestor de Kaggle...")
# Esto garantiza que siempre apunte a la carpeta correcta, tenga la versión que tenga
babble_root_path = kagglehub.dataset_download("artharking/babble-noise-mgdcc")
print(f"📂 Ruta real asignada: {babble_root_path}")

audio_buffers_list = []
max_archivos_a_combinar = 40 
sr_original = 16000
sr_objetivo = 22050

archivos_mat = 0
archivos_wav = 0

for root, dirs, files in os.walk(babble_root_path):
    for f in files:
        file_path = os.path.join(root, f)
        
        # --- INTENTO CON ARCHIVOS .MAT ---
        if f.lower().endswith('.mat'):
            archivos_mat += 1
            try:
                # MATLAB viejo
                mat_contents = scipy.io.loadmat(file_path)
                keys = [k for k in mat_contents.keys() if not k.startswith('__')]
                if keys:
                    audio_raw = mat_contents[keys[0]].flatten().astype(np.float32)
                    if len(audio_raw) > 0: audio_buffers_list.append(audio_raw)
            except:
                try:
                    # MATLAB v7.3 (HDF5)
                    with h5py.File(file_path, 'r') as f_h5:
                        keys = list(f_h5.keys())
                        if keys:
                            audio_raw = np.array(f_h5[keys[0]]).flatten().astype(np.float32)
                            if len(audio_raw) > 0: audio_buffers_list.append(audio_raw)
                except:
                    pass
                    
        # --- PLAN C: INTENTO CON ARCHIVOS .WAV ---
        elif f.lower().endswith('.wav'):
            archivos_wav += 1
            try:
                # Si el dataset resulta tener .wav, los cargamos directamente
                y, _ = librosa.load(file_path, sr=sr_original)
                if len(y) > 0: audio_buffers_list.append(y)
            except:
                pass

        # Frenamos si ya juntamos los 40 buffers
        if len(audio_buffers_list) >= max_archivos_a_combinar:
            break
            
    if len(audio_buffers_list) >= max_archivos_a_combinar:
        break

print("-" * 50)
print(f"Estadísticas de escaneo profundo:")
print(f" - Archivos .mat encontrados: {archivos_mat}")
print(f" - Archivos .wav encontrados: {archivos_wav}")
print(f" - Archivos extraídos con éxito: {len(audio_buffers_list)}")

if len(audio_buffers_list) > 0:
    print("\nConcatenando y remuestreando a 22050 Hz...")
    # Juntamos los pedazos
    babble_completo_16k = np.concatenate(audio_buffers_list)
    # Adaptamos la frecuencia de muestreo a tu pipeline (ESC-50)
    babble_audio_full = librosa.resample(babble_completo_16k, orig_sr=sr_original, target_sr=sr_objetivo)

    print(f"Babble noise estructurado correctamente")
    print(f"Duración total del murmullo: {len(babble_audio_full)/sr_objetivo:.2f} segundos.")
else:
    print("\nNo hay audios válidos en el dataset.")
    babble_audio_full = None

🔍 Solicitando la ruta exacta al gestor de Kaggle...
📂 Ruta real asignada: /Users/rociocaseres/.cache/kagglehub/datasets/artharking/babble-noise-mgdcc/versions/1
--------------------------------------------------
Estadísticas de escaneo profundo:
 - Archivos .mat encontrados: 40
 - Archivos .wav encontrados: 0
 - Archivos extraídos con éxito: 40

Concatenando y remuestreando a 22050 Hz...
Babble noise estructurado correctamente
Duración total del murmullo: 29.64 segundos.


In [9]:
# ==================================================
# 5. FUNCIONES DE DATA AUGMENTATION
# ==================================================

def add_white_noise(audio, noise_level=0.005):
    """Agrega ruido blanco Gaussiano estándar."""
    noise = np.random.randn(len(audio))
    return audio + noise_level * noise

def add_pink_noise(audio, noise_level=0.01):
    """Genera ruido rosa (decaimiento de 1/f en densidad espectral)."""
    # GenerO ruido blanco en el dominio de la frecuencia y aplicamos filtro 1/f
    white = np.random.randn(len(audio))
    fft_white = np.fft.rfft(white)
    # Filtro 1/sqrt(f) para la amplitud
    frequencies = np.maximum(np.fft.rfftfreq(len(audio)), 1e-10)
    f_filter = 1.0 / np.sqrt(frequencies)
    # Normalizo el filtro
    f_filter /= np.max(f_filter)
    fft_pink = fft_white * f_filter
    pink = np.fft.irfft(fft_pink, n=len(audio))
    # Normalizo amplitud
    pink = pink / np.max(np.abs(pink))
    return audio + noise_level * pink

def add_babble_noise(audio, babble_audio, noise_level=0.03):
    """Extrae un fragmento aleatorio del babble noise y lo mezcla."""
    if babble_audio is None:
        return audio

    # Si el audio de babble es más corto, se repite (un loop)
    if len(babble_audio) < len(audio):
        babble_audio = np.tile(babble_audio, int(np.ceil(len(audio) / len(babble_audio))))

    # Elegimos un punto de inicio aleatorio en el audio de babble
    start_idx = random.randint(0, len(babble_audio) - len(audio))
    babble_chunk = babble_audio[start_idx : start_idx + len(audio)]

    # Normalizamos el fragmento de babble
    babble_chunk = babble_chunk / (np.max(np.abs(babble_chunk)) + 1e-10)

    return audio + noise_level * babble_chunk



In [13]:
# ==========================================
# 2. TRANSFORMACIÓN Y DATA AUGMENTATION
# ==========================================
import os
import random
import librosa
import numpy as np
dataset_root_path = kagglehub.dataset_download("mmoreaux/environmental-sound-classification-50")
correct_audio_dir = None

# 1. BÚSQUEDA AUTOMÁTICA DE LA CARPETA
# Escaneamos el directorio raíz para encontrar dónde están realmente los .wav
for root, dirs, files in os.walk(dataset_root_path):
    if any(f.endswith('.wav') for f in files):
        correct_audio_dir = root
        break

# Verificamos si la encontró
if correct_audio_dir is None:
    raise FileNotFoundError(f"No se encontró ninguna carpeta con archivos .wav dentro de {dataset_root_path}")

X_list = []
y_list = []

print("Extrayendo características y aplicando Data Augmentation (esto puede demorar)...")

import pandas as pd

import os
import pandas as pd

# ==========================================
# 1.5 RECUPERAR EL DATAFRAME (BÚSQUEDA DINÁMICA)
# ==========================================
print("Buscando el archivo CSV de metadatos...")

csv_path = None

# Escaneamos todo el directorio descargado para encontrar dónde está el esc50.csv
for root, dirs, files in os.walk(dataset_root_path):
    if 'esc50.csv' in files:
        csv_path = os.path.join(root, 'esc50.csv')
        break

# Validamos si lo encontró
if csv_path is None:
    raise FileNotFoundError(f"❌ No se encontró el archivo 'esc50.csv' dentro de {dataset_root_path}")

print(f"✅ ¡Archivo CSV localizado en: {csv_path}!")

# Leemos el csv
df_completo = pd.read_csv(csv_path)

# Filtramos SOLO las 6 clases de tu PFI
clases_pfi = ['alarm', 'door_bell', 'cat', 'crying_baby', 'dog', 'shouting']
df_filtrado = df_completo[df_completo['category'].isin(clases_pfi)]

print(f"✅ DataFrame listo: {len(df_filtrado)} audios base listos para ser multiplicados.")

# Filtramos SOLO las 6 clases de tu PFI (timbre, alarmas, bebé, perro, gato, gritos)
clases_pfi = ['alarm', 'door_bell', 'cat', 'crying_baby', 'dog', 'shouting']
df_filtrado = df_completo[df_completo['category'].isin(clases_pfi)]

print(f"✅ DataFrame listo: {len(df_filtrado)} audios base encontrados para procesar.") 
for index, row in df_filtrado.iterrows():
    file_path = os.path.join(correct_audio_dir, row['filename'])
    categoria = row['category']

    try:
        # 1. Cargar el audio original
        y, sr = librosa.load(file_path, sr=22050)

        # 2. Generar las 4 variantes del audio en el dominio del tiempo
        audios_a_procesar = [
            y,                                           # Original, sin alteraciones
            add_white_noise(y),                          # Variante 1: Ruido Blanco
            add_pink_noise(y),                           # Variante 2: Ruido Rosa
            add_babble_noise(y, babble_audio_full)       # Variante 3: Murmullo de fondo
        ]

        # 3. Iterar sobre las 4 versiones para extraer sus características
        for audio_version in audios_a_procesar:
            
            # Extraer MFCCs
            mfccs = librosa.feature.mfcc(y=audio_version, sr=sr, n_mfcc=40)
            
            # Achatar a 2D (Vector plano de 40 números)
            mfccs_scaled_features = np.mean(mfccs.T, axis=0)

            # Guardar el vector y la misma etiqueta de categoría
            X_list.append(mfccs_scaled_features)
            y_list.append(categoria)

    except Exception as e:
        print(f"Error procesando {file_path}: {e}")

# Convertir las listas a matrices de Numpy
X = np.array(X_list)
y = np.array(y_list)

# Se agrega la dimensión de "canal" para que sktime (ROCKET) lo procese
X = np.expand_dims(X, axis=1)

print("-" * 50)
print("✅ Pipeline completado.")
print(f"Dimensiones de X finales: {X.shape}")
print(f"Total de audios procesados: {len(y)} (Originales x 4)")

Extrayendo características y aplicando Data Augmentation (esto puede demorar)...
Buscando el archivo CSV de metadatos...
✅ ¡Archivo CSV localizado en: /Users/rociocaseres/.cache/kagglehub/datasets/mmoreaux/environmental-sound-classification-50/versions/15/esc50.csv!
✅ DataFrame listo: 120 audios base listos para ser multiplicados.
✅ DataFrame listo: 120 audios base encontrados para procesar.
--------------------------------------------------
✅ Pipeline completado.
Dimensiones de X finales: (480, 1, 40)
Total de audios procesados: 480 (Originales x 4)


In [14]:
# ==========================================
# 3. CARGA, FILTRADO DEL DATASET (ESC-50) Y ENTRENAMIENTO
# ==========================================


# Paso las etiquetas de texto a números
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Separar en conjuntos de Entrenamiento y Prueba
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

#ROCKET
inicio = time.time()

# Instanciar y ajustar ROCKET
rocket = Rocket(num_kernels=5000, random_state=42)
rocket.fit(X_train)

# Transformar los audios a través de los kernels aleatorios
X_train_transform = rocket.transform(X_train)
X_test_transform = rocket.transform(X_test)

# Clasificador lineal final
classifier = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
classifier.fit(X_train_transform, y_train)

# Evaluar el modelo
score = classifier.score(X_test_transform, y_test)
fin = time.time()

print(f"Precisión (Test Accuracy): {score * 100:.2f}%")
print(f"Tiempo total de ejecución: {fin - inicio:.2f} segundos")

Precisión (Test Accuracy): 97.92%
Tiempo total de ejecución: 0.62 segundos


/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
